# Pruebas varias practica

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Sat May 16 20:00:27 2026

@author: lrobledo
"""
import numpy as np
import scipy.signal as sig
from pytc2.sistemas_lineales import analyze_sys, pretty_print_bicuad_omegayq, tf2sos_analog, pretty_print_SOS
from pytc2.general import print_subtitle
this_order = 5
this_ripple = 0.25
eps = np.sqrt( 10**(this_ripple/10) - 1 )
den = np.array([1, 0, 640/256, 0, 560/256, 0, 200/256, 0 , 25/256, 0 , -1/256/ eps**2])
np.roots(den)
cheby_roots = np.roots(den)
cheby_roots = cheby_roots[ np.real(cheby_roots) < 0 ]
cheby_roots
p= cheby_roots 
k=-np.sqrt(1/256/ eps**2)
z = np.array([])  # Chebyshev I no tiene ceros
num, den = sig.zpk2tf(z,p,k)
mi_tf = sig.TransferFunction(num,den)
from pytc2.sistemas_lineales import analyze_sys, pretty_print_bicuad_omegayq, tf2sos_analog, pretty_print_SOS
analyze_sys( mi_tf, sys_name= "chby 1" )


this_sos = tf2sos_analog(num, den)
pretty_print_SOS(this_sos, mode='omegayq')
from pytc2.sistemas_lineales import pretty_print_lti
pretty_print_lti(num,den)

#cheby verificado por pyton para ver que den la misma funcion

z, p, k = sig.cheb1ap(
    this_order,
    this_ripple
)

num, den = sig.zpk2tf(z,p,k)

mi_tf = sig.TransferFunction(
    num,
    den
)

this_sos = tf2sos_analog(num, den)
pretty_print_SOS(this_sos, mode='omegayq')
pretty_print_lti(num,den)

analyze_sys(
    mi_tf,
    sys_name='Cheby orden 5'
)


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Sat May 16 20:00:27 2026

@author: lrobledo
"""
# Librerías externas NumPy, SciPy y Matplotlib
from scipy.signal import TransferFunction
import matplotlib.pyplot as plt
import numpy as np
from pytc2.sistemas_lineales import pretty_print_lti

from pytc2.sistemas_lineales import analyze_sys, pretty_print_bicuad_omegayq, tf2sos_analog, pretty_print_SOS

from pytc2.general import print_subtitle
# Librería de TC2, esta la vas a usar mucho
from pytc2.sistemas_lineales import pzmap, GroupDelay, bodePlot

from sympy import *
this_order = 5
this_ripple = 0.25
w0 = 1.692
q  = 7.975
n=400867.22
num=np.array([w0**2/q**2,0,0])
den = np.array([1, np.sqrt(2)*w0*n/q, (2+w0**2/q**2)*n**2, np.sqrt(2)*w0*n**3/q, n**4])
my_tf = TransferFunction(num, den)

plt.close('all')

my_tf = TransferFunction(num, den)

bodePlot(my_tf, fig_id=1, filter_description = 'Cn=5')

pzmap(my_tf, fig_id=2, filter_description = 'Cn=5')

GroupDelay(my_tf, fig_id=3, filter_description = 'Cn=5')

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import TransferFunction, freqresp

# --- Definición del sistema (tal como ya lo tenías, en rad/s) ---
w0 = 1.692
q  = 7.975
n=400867.22
num = np.array([(w0*n)**2/q**2, 0, 0])
den = np.array([1, np.sqrt(2)*w0*n/q, (2+w0**2/q**2)*n**2, np.sqrt(2)*w0*n**3/q, n**4])
my_tf = TransferFunction(num, den)

# ==========================================================
# Respuesta en frecuencia, calculada directamente en Hz
# ==========================================================

# Rango de frecuencias a graficar (ajustar según tu filtro)
f_hz = np.logspace(2, 6, 20000)   # 100 Hz a 1 MHz
w_rad = 2*np.pi*f_hz              # scipy trabaja en rad/s internamente

_, H = freqresp(my_tf, w=w_rad)
mag_db   = 20*np.log10(np.abs(H))
phase_deg = np.unwrap(np.angle(H)) * 180/np.pi

# ==========================================================
# Detección del pico y del ancho de banda a -3 dB
# ==========================================================

idx_peak = np.argmax(mag_db)
f_peak   = f_hz[idx_peak]
mag_peak = mag_db[idx_peak]
nivel_3db = mag_peak - 3

def find_crossing(f, mag, level, idx_peak, direction):
    """Busca el cruce de -3dB interpolando en escala logarítmica de frecuencia."""
    idxs = np.arange(idx_peak, -1, -1) if direction == 'left' else np.arange(idx_peak, len(mag))
    for i in range(len(idxs) - 1):
        i0, i1 = idxs[i], idxs[i+1]
        if (mag[i0] - level) * (mag[i1] - level) <= 0:
            logf0, logf1 = np.log10(f[i0]), np.log10(f[i1])
            m0, m1 = mag[i0], mag[i1]
            logf_cross = logf0 + (level - m0) * (logf1 - logf0) / (m1 - m0)
            return 10**logf_cross
    return None

f_low  = find_crossing(f_hz, mag_db, nivel_3db, idx_peak, 'left')
f_high = find_crossing(f_hz, mag_db, nivel_3db, idx_peak, 'right')

BW = f_high - f_low
f_centro_geo = np.sqrt(f_low * f_high)   # centro "ideal" de un pasabanda simétrico
Q_medido = f_peak / BW

print("------------------------------------------------")
print("VERIFICACIÓN DEL PASABANDA")
print("------------------------------------------------")
print(f"f0 de diseño (w0/2π)     = {w0/(2*np.pi):.2f} Hz")
print(f"Frecuencia del pico      = {f_peak:.2f} Hz")
print(f"Magnitud en el pico      = {mag_peak:.3f} dB")
print(f"f baja  (-3 dB)          = {f_low:.2f} Hz")
print(f"f alta  (-3 dB)          = {f_high:.2f} Hz")
print(f"Ancho de banda (BW)      = {BW:.2f} Hz")
print(f"Centro geométrico        = {f_centro_geo:.2f} Hz  (debería ≈ f pico si está bien centrado)")
print(f"Q medido = f_pico/BW     = {Q_medido:.3f}   (Q diseño = {q})")

desvio_centro = abs(f_peak - f_centro_geo) / f_centro_geo * 100
print(f"\nDesvío del centro respecto al ideal: {desvio_centro:.3f} %")
if desvio_centro < 1:
    print("✓ El pasabanda está bien centrado (<1% de desvío)")
else:
    print("✗ El pasabanda presenta asimetría notable")

# ==========================================================
# Gráfico de Bode en Hz
# ==========================================================


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

ax1.semilogx(f_hz, mag_db, color='#1f77b4', linewidth=1.5)
ax1.axhline(nivel_3db, color='gray', linestyle='--', linewidth=1, label='-3 dB')
ax1.axvline(f_peak, color='green', linestyle=':', label=f'f pico = {f_peak:.0f} Hz')
ax1.axvline(f_low,  color='red', linestyle=':', label=f'f baja = {f_low:.0f} Hz')
ax1.axvline(f_high, color='red', linestyle=':', label=f'f alta = {f_high:.0f} Hz')
ax1.set_ylabel('Magnitud (dB)', fontsize=12)
ax1.set_title('Diagrama de Bode - Magnitud (en Hz)', fontsize=14)
ax1.grid(True, which='both', linestyle='--', alpha=0.6)
ax1.legend(loc='lower center')

ax2.semilogx(f_hz, phase_deg, color='#d62728', linewidth=1.5)
ax2.axvline(f_peak, color='green', linestyle=':')
ax2.set_xlabel('Frecuencia (Hz)', fontsize=12)
ax2.set_ylabel('Fase (°)', fontsize=12)
ax2.grid(True, which='both', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Wed May 13 19:34:17 2026

@author: lrobledo
"""

# Librerías externas NumPy, SciPy y Matplotlib
from scipy.signal import TransferFunction
import matplotlib.pyplot as plt
import numpy as np
from pytc2.sistemas_lineales import pretty_print_lti

from pytc2.sistemas_lineales import analyze_sys, pretty_print_bicuad_omegayq, tf2sos_analog, pretty_print_SOS

from pytc2.general import print_subtitle
# Librería de TC2, esta la vas a usar mucho
from pytc2.sistemas_lineales import pzmap, GroupDelay, bodePlot

from sympy import *

# Variables simbólicas
V1, Vo, Vx, s, C, R1, R2, R3 = symbols('V1 Vo Vx s C R1 R2 R3')

# Ecuaciones
eq1 = Eq(V1/R1, -Vo*(s*C + 1/R2)-Vx/R3)
eq2 = Eq(Vo/R3, Vx*s*C)

# Resolver sistema
sol = solve((eq1, eq2), (Vo, Vx))[Vo]/V1


#no es el mismo
#parcial del curso de los lunes

Q=7.975
psy= 0.122
W=140016.9
#desnormalizada
#num = [(W**2)/(Q**2),0,0]   # 1
#den = [1,1.41*W/(Q),(2+1/Q**2)*W**2, (1.41*W**3)/(Q),W**4]       # s + k

#normalizada
num = [(1)/(Q**2),0,0]   # 1
den = [1,1.41/(Q),(2+1/Q**2), (1.41)/(Q),1]


my_tf = TransferFunction(num, den)

plt.close('all')

my_tf = TransferFunction(num, den)

bodePlot(my_tf, fig_id=1, filter_description = 'Cn=5')

pzmap(my_tf, fig_id=2, filter_description = 'Cn=5')

GroupDelay(my_tf, fig_id=3, filter_description = 'Cn=5')

this_sos = tf2sos_analog(num, den)
pretty_print_SOS(this_sos, mode='omegayq')
pretty_print_lti(num,den)

In [ ]:
# Variables simbólicas
V1, Vo, Vx, Vy, s, C, R, R1, R3, R4, R5  = symbols('V1 Vo Vx Vy s C R R1 R3 R4 R5')

# Ecuaciones
eq1 = Eq((V1-Vx)/R, Vx*s*C+(Vx-Vy)/R1)
eq2 = Eq((Vy-Vx)*-s*C, (Vx-Vo)/R3)
eq3 = Eq((Vo-Vx)*R4, Vx/R5)


# Resolver sistema
sol = solve((eq1, eq2, eq3), (Vo, Vx, Vy), dict=True)
H = simplify(sol[0][Vo]/V1)

pretty_print(H)